In [0]:
# Step 1: Set up the streaming paths
catalog = spark.sql("SELECT current_catalog()").first()[0]
schema_name = "retail_lakehouse"

volume_path = f"/Volumes/{catalog}/{schema_name}/retail_landing"
input_path = f"{volume_path}/streaming_orders"
checkpoint_path = f"{volume_path}/checkpoints/retail_streaming"
table_name = f"{catalog}.{schema_name}.streaming_orders_bronze"

# Create the folders used by the streaming pipeline
dbutils.fs.mkdirs(input_path)
dbutils.fs.mkdirs(checkpoint_path)

print("Streaming input:", input_path)
print("Checkpoint:", checkpoint_path)
print("Target table:", table_name)

In [0]:
# Step 2: Define the structure of incoming order files
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType
)

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("status", StringType(), True),
    StructField("region", StringType(), True)
])

print("Order schema is ready.")

In [0]:
# Step 3: Create a sample batch of incoming orders
sample_orders = [
    ("S3001", "2026-10-10", "C030", "P101", 2, 499.0, "Completed", "West"),
    ("S3002", "2026-10-10", "C031", "P102", 1, 999.0, "Completed", "South")
]

sample_df = spark.createDataFrame(sample_orders, schema=orders_schema)

# Save this batch in its own folder so streaming can discover the files
sample_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{input_path}/batch_001")

display(sample_df)

In [0]:
# Step 4: Read incoming CSV files using Structured Streaming
streaming_orders_df = (
    spark.readStream
    .format("csv")
    .schema(orders_schema)
    .option("header", True)
    .option("recursiveFileLookup", "true")
    .load(input_path)
)

# display() previews streams in complete output mode, which only works
# with streaming aggregations. Stream to a memory sink in append mode instead
# and display the resulting table.
import time

# The memory sink cannot recover from a previous checkpoint, so give the
# one-shot preview query a fresh checkpoint location on every run.
preview_checkpoint = f"{checkpoint_path}_preview_{int(time.time())}"

preview_query = (
    streaming_orders_df.writeStream
    .format("memory")
    .queryName("orders_stream_preview")
    .outputMode("append")
    .option("checkpointLocation", preview_checkpoint)
    .trigger(availableNow=True)
    .start()
)
preview_query.awaitTermination()

display(spark.table("orders_stream_preview"))

In [0]:
# Step 5: Ingest available files into a separate Bronze Delta table
stream_query = (
    streaming_orders_df.writeStream
    .format("delta")
    .outputMode("append")
    # This query needs its own checkpoint location: the previously shared
    # one holds state from a different streaming query and cannot be resumed.
    .option("checkpointLocation", f"{checkpoint_path}_bronze")
    .trigger(availableNow=True)
    .toTable(table_name)
)

stream_query.awaitTermination()

print("Streaming ingestion finished.")

In [0]:
# Verify the streamed orders
from pyspark.sql import functions as F

result_df = spark.table(table_name)

result_df.select(
    F.count("*").alias("total_rows"),
    F.countDistinct("order_id").alias("unique_order_ids")
).show()

display(result_df.orderBy("order_id"))

In [0]:
# Step 7: Check for missing IDs and invalid quantities
from pyspark.sql import functions as F

quality_summary = result_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("order_id").isNull(), 1).otherwise(0)).alias("missing_order_ids"),
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("invalid_quantities"),
    F.countDistinct("order_id").alias("unique_order_ids")
)

display(quality_summary)